# Libraries


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Variables

**Duty cycle**

In [ ]:
#10797420 so 20 % 50 TODO check this
timeSleepSeconds = (20 % 50 + 5) / 10.0 
timeSleepMicroseconds = timeSleepSeconds * 1000000
print(timeSleepMicroseconds, "microseconds")

**Battery**

In [ ]:
#10797420 so 7420 TODO check this
JBattery= 7420%5000+15000
print(JBattery, "Joule")

# Dataset Analysis

**Structure of the dataset**

In [ ]:
df_sleep = pd.read_csv('data/deep_sleep.csv')
df_sender = pd.read_csv('data/sender.csv')
df_sensor = pd.read_csv('data/sensor-read.csv')

**Description of the dataset**

In [ ]:
print(df_sleep.describe())
print(df_sender.describe())
print(df_sensor.describe())

# Average cycle power consumption



Computing the average power consuption during each functional state requires filtering the datapoints according to empirical observation of the plotted data.

| deep sleep | idle              | WiFi on  |
|------------|-------------------|----------|
| < 100 mW   | >200 mW, < 300 mW | > 400 mW | 

In [ ]:
df_sleep.plot(
    xlabel="time [s]",
    ylabel="Power [mW]",
    title="Power consumption (Deep Sleep)",
    kind="line",
    legend=False,
)

In [ ]:
deep_sleep_mean = round(df_sleep['Data'][df_sleep['Data'] < 100].mean(), 3)
idle_mean = round(df_sleep['Data'][df_sleep['Data'] > 200][df_sleep['Data'] < 300].mean(), 3)
tx_mean = round(df_sleep['Data'][df_sleep['Data'] > 400].mean(), 3)

print ("Deep Sleep Mean:", deep_sleep_mean, "mW")
print ("Idle Mean:", idle_mean, "mW")
print ("WiFi on Mean:", tx_mean, "mW")


# Data trasmissions consumption avgs


trasmission data from sender.csv, analysis on power consumption at 2dBm, 19.5dBm and idle

In [ ]:
df_sender.plot(
    xlabel="time [s]",
    ylabel="Power [mW]",
    title="Power consumption (Sender)",
    kind="line",
    legend=False,
)

In [ ]:
idle_wifi = round(df_sender['Data'][df_sender['Data'] < 610].mean(), 3)
tx2db_wifi = round(df_sender['Data'][df_sender['Data'] > 610][df_sender['Data'] < 630].mean(), 3)
tx19db_wifi = round(df_sender['Data'][df_sender['Data'] > 640].mean(), 3)
print ("Idle Mean:", idle_wifi, "mW")
print ("Tx 2dBm Mean:", tx2db_wifi, "mW")
print ("Tx 19.5dBm Mean:", tx19db_wifi, "mW")

# Sensor reading power consumption avg

We assume that the data regarding the sensor consumption refers to the total energy consumed by all the sensors connected to the device, which in our case are the motion and luminosity sensors.

In [ ]:
df_sensor.plot(
    xlabel="time [s]",
    ylabel="Power [mW]",
    title="Power consumption (Sensor)",
    kind="line",
    legend=False,
)

In [ ]:
sensor_off_mean = round(df_sensor['Data'][df_sensor['Data'] < 300].mean(), 3)
sensor_on_mean = round(df_sensor['Data'][df_sensor['Data'] > 300].mean(), 3)
print ("Sensor off Mean:", sensor_off_mean, "mW")
print ("Sensor on Mean:", sensor_on_mean, "mW")

# Energy computation


In [ ]:
states = [
    "Deep Sleep",
    "Idle",
    "Tx",
    "Sensor On",
    "Sensor Off",
    "WiFi idle",
    "WiFi Tx 2dBm",
    "WiFi Tx 19.5dBm",
]

power_avgs = [
    deep_sleep_mean,
    idle_mean,
    tx_mean,
    sensor_on_mean,
    sensor_off_mean,
    idle_wifi,
    tx2db_wifi,
    tx19db_wifi,
]

sorted_power_data = zip(states, power_avgs)
# print(sorted_power_data)
print("\nAverage Power Consumption for each state:")
for state, power in sorted_power_data:
    print(f"{state}: {power} mW")

In [ ]:
sorted_power_data = sorted(zip(power_avgs, states))
sorted_power_values, sorted_states = zip(*sorted_power_data)

plt.figure(figsize=(10, 5))
plt.bar(
    sorted_states,
    sorted_power_values,
    color=["green", "blue", "blue", "red", "orange", "purple", "purple", "black"],
)

plt.ylabel("Power [mW]")
plt.title("Power consuming")
plt.xticks(rotation=45)

plt.show()

In our case we don't consume Transmission power at 19.5 dBm, due the fact that we use the component only at 2dBm

# Time Estimation of our simulation

In [ ]:
time_dataset = pd.read_csv("data/timings_from_simulation.csv")
print("Dataset shape (working cycle of the esp): \n", time_dataset.head(7))

time_dataset["Duration"] = time_dataset["Timestamp"].diff()
time_dataset.dropna(inplace=True)


print(time_dataset)
diff_labels = [
    "idle time",
    "sensor reading time",
    "wifi init time",
    "transmission [2 dBm] time",
    "wifi shutdown time",
    "entering deep sleep time"
]

time_results = dict(zip(diff_labels, time_dataset["Duration"].values[:len(diff_labels)]))

print("\n\nAverage time for each operational state: \n")
for key, value in time_results.items():
    print(f"{key}: {value:.2f} us")

plt.figure(figsize=(8, 5))
plt.bar(time_results.keys(), time_results.values(), color='orange') # type: ignore
plt.ylabel("Time (us)")
plt.title("Average duration of each operational state")
plt.xticks(rotation=45)
plt.show()


# Energy consumption for each state

In [ ]:
energy_idle = (idle_mean * time_results["idle time"]) / 1000000000 
energy_sensor_reading = (sensor_on_mean * time_results["sensor reading time"]) / 1000000000
energy_wifi_init = (idle_wifi * time_results["wifi init time"]) / 1000000000
energy_wifi_tx = (tx2db_wifi * time_results["transmission [2 dBm] time"]) / 1000000000
energy_wifi_shutdown = (idle_wifi * time_results["wifi shutdown time"]) / 1000000000
# timeSleepMicroseconds is the time spent in deep sleep
energy_deep_sleep = (deep_sleep_mean * timeSleepMicroseconds) / 1000000000 

print("\n\nEnergy consumption for each state: \n")
print(f"Idle: {energy_idle:.6f} J")
print(f"Sensor Reading: {energy_sensor_reading:.6f} J")
print(f"WiFi Init: {energy_wifi_init:.6f} J")
print(f"WiFi Tx: {energy_wifi_tx:.6f} J")
print(f"WiFi Shutdown: {energy_wifi_shutdown:.6f} J")
print(f"Deep Sleep: {energy_deep_sleep:.6f} J")

tot_energy_wifi = energy_idle + energy_sensor_reading + energy_wifi_init + energy_wifi_tx + energy_wifi_shutdown + energy_deep_sleep
tot_time_micorseconds = time_results["idle time"] + time_results["sensor reading time"] + time_results["wifi init time"] + time_results["transmission [2 dBm] time"] + time_results["wifi shutdown time"] + timeSleepMicroseconds
tot_time = tot_time_micorseconds / 1000000

In [ ]:
# plotting energy consumption
energy_states = [
    "Idle",
    "Sensor Reading",
    "WiFi Init",
    "WiFi Tx",
    "WiFi Shutdown",
    "Deep Sleep",
]

energy_values = [
    energy_idle,
    energy_sensor_reading,
    energy_wifi_init,
    energy_wifi_tx,
    energy_wifi_shutdown,
    energy_deep_sleep,
]

plt.figure(figsize=(10, 5))
plt.bar(
    energy_states,
    energy_values,
    color=["blue", "red", "purple", "black", "purple", "green"],
)
plt.ylabel("Energy (Joules)")
plt.title("Energy consumption for each state")
plt.xticks(rotation=45)
plt.show()

# Battery life computation

In [ ]:
duty_cycles = JBattery/tot_energy_wifi
print(f"duty cycles: {duty_cycles:.0f}")

battery_life = duty_cycles*tot_time/3600
print(f"battery life: {battery_life:.2f} h")

# Possible Improvements

write improvements here:

use wi fi only when motion detected or light is above a certain treshold (luminosity threshold if sensor is outside, otherwise only motion detection maybe)

wake up every 4/5 seconds so we have longer sleeps (may impact UX but may save energy)

alternative techs since a couple of days of battery duration isnt really acceptable for the use case

if we use wi fi as specified above then we can have an extimation of the battery life given that we will turn on the wi fi for 20/25% of the battery lifetime considering an outdoor sensor (maybe more 40% for an indoor one without the luminosity threshold) since we may assume half the day with low light and the other half with good light, and that we will have motion detected for 50% of the time (which is a very high estimation but we want to be sure that we are not overestimating the battery life)

In [ ]:
# battery life estimation not using any wi fi
tot_energy_no_wifi = energy_idle + energy_sensor_reading + energy_deep_sleep
duty_cycles_no_wifi = JBattery/tot_energy_no_wifi
tot_time_no_wifi_microseconds = time_results["idle time"] + time_results["sensor reading time"] + timeSleepMicroseconds
tot_time_no_wifi = tot_time_no_wifi_microseconds / 1000000
battery_life_no_wifi = duty_cycles_no_wifi*tot_time_no_wifi/3600
print(f"battery life without wifi: {battery_life_no_wifi:.2f} h")

In [ ]:
# weighted average of energy consumption considering 25% of the time we use wi fi and 75% we dont use it
tot_energy_mixed = 0.25 * tot_energy_wifi + 0.75 * tot_energy_no_wifi
duty_cycles_mixed = JBattery/tot_energy_mixed
tot_mixed_time_microseconds = 0.25 * tot_time_micorseconds + 0.75 * tot_time_no_wifi_microseconds
tot_time_mixed = tot_mixed_time_microseconds / 1000000
battery_life_mixed = duty_cycles_mixed*tot_time_mixed/3600
print(f"battery life with 25% wifi usage: {battery_life_mixed:.2f} h")

In [ ]:
# percentage of battery life gain
battery_life_gain = ((battery_life_mixed - battery_life) / battery_life) * 100
print(f"battery life gain with 25% wifi usage: {battery_life_gain:.2f} %")
# hours gained
hours_gained = battery_life_mixed - battery_life
print(f"hours gained with 25% wifi usage: {hours_gained:.2f} h")